In [1]:
# Basic imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier
import spacy

RND = 42

In [2]:
# Load the expanded dataset
DATA_PATH = '../resources/dataset/synonym_youtoxic_english_1000.csv'
assert os.path.exists(DATA_PATH), f"Data file not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print('Loaded', DATA_PATH, 'shape=', df.shape)
df.head()

Loaded ../resources/dataset/synonym_youtoxic_english_1000.csv shape= (3751, 16)


,Unnamed: 0,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement embody not train to shoot to a...,True,True,False,False,False,False,False,False,False,False,False,False
3,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement make up not trained to shoot t...,True,True,False,False,False,False,False,False,False,False,False,False
4,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Legal philosophy enforcement is not trained to...,True,True,False,False,False,False,False,False,False,False,False,False


colums using:    
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist'

In [3]:
# Identify label columns (common names used in the original notebook)
target_cols = [
    'IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist'
]
# Keep only the labels that are present in the dataset
present_targets = [c for c in target_cols if c in df.columns]
if len(present_targets) == 0:
    raise ValueError("No expected target columns found in the uploaded CSV. Please ensure the file contains at least one of: " + ','.join(target_cols))
print('Using target columns:', present_targets)

Using target columns: ['IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist']


In [4]:
# Safe text column detection
text_col = None
for candidate in ['text','Text','comment_text','comment','commentText']:
    if candidate in df.columns:
        text_col = candidate
        break
if text_col is None:
    raise ValueError("No text column found in data (expected one of: text, Text, comment_text, comment, commentText).")
print('Using text column:', text_col)
df[text_col] = df[text_col].astype(str)

Using text column: Text


In [5]:
# Load SpaCy (may be cached in the environment) and define lemmatizer
try:
    nlp = spacy.load('en_core_web_sm')
except Exception as e:
    import subprocess
    subprocess.check_call(['python','-m','spacy','download','en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

def spacy_lemmatize(series):
    # simple batch lemmatizer (works reasonably fast for small datasets)
    texts = []
    for doc in nlp.pipe(series.astype(str).tolist(), batch_size=50, disable=['ner','parser']):
        lem = ' '.join([t.lemma_.lower() for t in doc if (not t.is_punct) and (not t.is_space)])
        texts.append(lem)
    return pd.Series(texts)


In [6]:
# Create lemmatized text column and train/test split
df['lemmatized'] = spacy_lemmatize(df[text_col])
# Multi-label: split while attempting to stratify by the first present target if possible
stratify_col = None
if present_targets and present_targets[0] in df.columns:
    stratify_col = df[present_targets[0]]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=RND, stratify=stratify_col if stratify_col is not None else None)
print('Train shape:', train_df.shape, 'Test shape:', test_df.shape)



Train shape: (3000, 17) Test shape: (751, 17)


In [7]:
# TF-IDF on lemmatized text
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english')
X_train = tfidf.fit_transform(train_df['lemmatized'])
X_test = tfidf.transform(test_df['lemmatized'])
print('TF-IDF shapes:', X_train.shape, X_test.shape)


TF-IDF shapes: (3000, 2000) (751, 2000)


In [8]:
# Prepare multi-label y matrices (only for present targets)
y_train = train_df[present_targets].astype(int)
y_test = test_df[present_targets].astype(int)
print('y shapes:', y_train.shape, y_test.shape)


y shapes: (3000, 6) (751, 6)


## Conservative model: TruncatedSVD + LogisticRegression (OOF-guided)
We rerun the conservative pipeline that was used to minimize the OOF-to-test overfitting gap. This will compute OOF predictions (cross-validated) and a final test evaluation.

In [9]:
# Fit SVD on the TF-IDF and reduce to 1 component (same aggressive underfitting that produced low gap previously)
svd = TruncatedSVD(n_components=1, random_state=RND)
X_tr_red = svd.fit_transform(X_train)
X_te_red = svd.transform(X_test)
print('Reduced shapes:', X_tr_red.shape, X_te_red.shape)

Reduced shapes: (3000, 1) (751, 1)


In [10]:
# Train per-label logistic regressions and compute OOF predictions
oof_preds = np.zeros((len(train_df), len(present_targets)), dtype=int)
oof_probas = np.zeros((len(train_df), len(present_targets)))
final_clfs = {}
kf = KFold(n_splits=5, shuffle=True, random_state=RND)
for i, lab in enumerate(present_targets):
    y_col = y_train[lab].values
    clf = LogisticRegression(C=0.1, class_weight='balanced', max_iter=200, random_state=RND)
    # OOF probability predictions using cross_val_predict (method='predict_proba')
    try:
        proba_oof = cross_val_predict(clf, X_tr_red, y_col, cv=kf, method='predict_proba')[:, 1]
    except Exception as e:
        # fallback to predict if proba not available from wrapper
        print('cross_val_predict failed for', lab, '->', e)
        proba_oof = cross_val_predict(clf, X_tr_red, y_col, cv=kf, method='predict')
    oof_probas[:, i] = proba_oof
    oof_preds[:, i] = (proba_oof >= 0.5).astype(int)
    # Train final classifier on full reduced training set
    clf_full = LogisticRegression(C=0.1, class_weight='balanced', max_iter=200, random_state=RND)
    clf_full.fit(X_tr_red, y_col)
    final_clfs[lab] = clf_full
    print('Trained logistic for', lab)

# Compute OOF and test metrics
oof_f1 = f1_score(y_train.values, oof_preds, average='macro')
# Predict test using final_clfs
test_probas = np.column_stack([final_clfs[lab].predict_proba(X_te_red)[:,1] for lab in present_targets])
test_preds = (test_probas >= 0.5).astype(int)
test_f1 = f1_score(y_test.values, test_preds, average='macro')
overfit_gap = (oof_f1 - test_f1) / oof_f1 if oof_f1 != 0 else None
print(f"OOF F1 (macro): {oof_f1:.6f}\nTest F1 (macro): {test_f1:.6f}\nOverfit gap: {overfit_gap:.6f}")

Trained logistic for IsToxic
Trained logistic for IsAbusive
Trained logistic for IsProvocative
Trained logistic for IsObscene
Trained logistic for IsHatespeech
Trained logistic for IsRacist
OOF F1 (macro): 0.481214
Test F1 (macro): 0.479622
Overfit gap: 0.003308


## XGBoost baseline (One-vs-Rest wrapper)
Train a comparable XGBoost pipeline for comparison. This uses the same SpaCy TF‑IDF features and produces OOF probabilities for a fair comparison where possible.

In [11]:
# XGBoost with OneVsRest for multi-label and OOF probabilities where possible
xgb_params = dict(n_estimators=200, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss', random_state=RND)
xgb = OneVsRestClassifier(XGBClassifier(**xgb_params), n_jobs=1)
# OOF probabilities via cross_val_predict are heavy; we attempt a fold-wise approach for probabilities
probas_oof = np.zeros((len(train_df), len(present_targets)))
preds_test = np.zeros((len(test_df), len(present_targets)), dtype=int)
for i, lab in enumerate(present_targets):
    y_col = y_train[lab].values
    # simple per-label XGB using KFold to create OOF probs
    oof_proba_label = np.zeros(len(train_df))
    test_proba_label = np.zeros(len(test_df))
    for fold, (tr_idx, val_idx) in enumerate(KFold(n_splits=5, shuffle=True, random_state=RND).split(X_train)):
        X_tr_fold = X_train[tr_idx]
        X_val_fold = X_train[val_idx]
        y_tr = y_col[tr_idx]
        y_val = y_col[val_idx]
        m = XGBClassifier(**xgb_params)
        # Some xgboost versions don't accept early_stopping_rounds via the sklearn wrapper;
        # omit it for compatibility and rely on fixed n_estimators instead.
        m.fit(X_tr_fold, y_tr, eval_set=[(X_val_fold, y_val)], verbose=False)
        oof_proba_label[val_idx] = m.predict_proba(X_val_fold)[:, 1]
        test_proba_label += m.predict_proba(X_test)[:, 1] / 5.0
    probas_oof[:, i] = oof_proba_label
    preds_test[:, i] = (test_proba_label >= 0.5).astype(int)
    print('Finished XGB label', lab)

xgb_oof_f1 = f1_score(y_train.values, (probas_oof >= 0.5).astype(int), average='macro')
xgb_test_f1 = f1_score(y_test.values, preds_test, average='macro')
xgb_overfit_gap = (xgb_oof_f1 - xgb_test_f1) / xgb_oof_f1 if xgb_oof_f1 != 0 else None
print(f"XGB OOF F1: {xgb_oof_f1:.6f}\nXGB Test F1: {xgb_test_f1:.6f}\nXGB Overfit gap: {xgb_overfit_gap:.6f}")



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:21:07] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:22:11] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:23:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:24:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/wor

Finished XGB label IsToxic


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:26:18] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:27:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:28:18] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:29:22] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/wor

Finished XGB label IsAbusive


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:32:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:33:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:34:33] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:35:35] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/wor

...

Finished XGB label IsObscene


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:42:23] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:43:17] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:44:16] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:46:44] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/wor

Finished XGB label IsHatespeech


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:52:00] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:54:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:57:02] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:59:38] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/wor

Finished XGB label IsRacist
XGB OOF F1: 0.771017
XGB Test F1: 0.746079
XGB Overfit gap: 0.032344


In [12]:
# Save summary results and models
out_dir = '/workspaces/proyecto10-grupo5/resources/models'
os.makedirs(out_dir, exist_ok=True)
# Save vectorizer and SVD and logistic classifiers
joblib.dump(tfidf, os.path.join(out_dir, 'final_tfidf_spacy_synonym_1000.pkl'))
joblib.dump(svd, os.path.join(out_dir, 'final_svd_synonym_1000.pkl'))
joblib.dump(final_clfs, os.path.join(out_dir, 'final_logreg_clfs_synonym_1000.pkl'))
# Save XGBoost OOF arrays for later inspection
np.savez_compressed(os.path.join(out_dir, 'final_xgb_oof_synonym_1000.npz'), probas_oof=probas_oof)
# Write a JSON summary
summary = {
    'conservative': {
        'oof_f1': float(oof_f1),
        'test_f1': float(test_f1),
        'overfit_gap': float(overfit_gap) if overfit_gap is not None else None
    },
    'xgboost': {
        'oof_f1': float(xgb_oof_f1),
        'test_f1': float(xgb_test_f1),
        'overfit_gap': float(xgb_overfit_gap) if xgb_overfit_gap is not None else None
    },
    'n_train': int(len(train_df)),
    'n_test': int(len(test_df)),
    'targets': present_targets
}
with open(os.path.join('/workspaces/proyecto10-grupo5/eda/json/summary_synonym_1000.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved results to', out_dir)
print(json.dumps(summary, indent=2))



Saved results to /workspaces/proyecto10-grupo5/resources/models
{
  "conservative": {
    "oof_f1": 0.48121387705515156,
    "test_f1": 0.4796220890795775,
    "overfit_gap": 0.0033078596679614117
  },
  "xgboost": {
    "oof_f1": 0.7710168648276644,
    "test_f1": 0.7460787345874986,
    "overfit_gap": 0.032344467906989145
  },
  "n_train": 3000,
  "n_test": 751,
  "targets": [
    "IsToxic",
    "IsAbusive",
    "IsProvocative",
    "IsObscene",
    "IsHatespeech",
    "IsRacist"
  ]
}
